## Delayed VLM trajectory advice + score fusion (playground)

This notebook is a *tiny, non-visual*, closed-loop simulator to study delayed VLM trajectory advice and “score fusion” against a fast local planner.

### What you can vary
- **task**: `forward`, `left_turn`, `right_turn`
- **trajectory set**: static candidate library (e.g. `static_candidates_k12.json`)
- **controller**: `endpoint_pd` (baseline) vs `pure_pursuit`
- **VLM delay**: seconds
- **policy**:
  - `local_only`: always pick planner argmax
  - `vlm_hold`: execute the last VLM trajectory directly (stale)
  - `score_fusion`: re-rank planner candidates using similarity to the stale VLM trajectory

### Robot / dynamics assumptions (IMPORTANT)
These are the assumptions used *everywhere in this notebook* (same as `slow_brain_fast_planner.control.tracking` / `slow_brain_fast_planner.control.delayed_vlm_score_fusion`):

- **Plant model**: ideal diff-drive / unicycle 
  - state: \((x, y, \psi)\)
  - controls: forward speed \(v\) and yaw rate \(\omega\)
- **Perfect state estimate**: no GPS/odom noise, no drift, no delay
- **No actuation latency**: controls apply instantly (but are rate-limited)
- **No slip / terrain effects**
- **No obstacles / footprint**: we measure *tracking* + *selection stability* only
- **Constraints are proxies** (tune these to match your robot):
  - `max_v` [m/s], `max_w` [rad/s]
  - `max_a` [m/s²], `max_alpha` [rad/s²]
  - `max_lat_accel` [m/s²] used by pure pursuit to cap speed on curvature

### Coordinate conventions
- Trajectory points are in **robot frame** at planning time: **x forward, y left**.
- Plots use **(y_left, x_forward)** so “forward” points upward.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np

# Optional (nice tables); if missing, the notebook still works.
try:
    import pandas as pd
except Exception:  # noqa: BLE001
    pd = None

import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src").exists():
            return p
    return start


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("repo:", REPO)
print("python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", getattr(pd, "__version__", None))
print("matplotlib:", plt.matplotlib.__version__)

In [ ]:
from slow_brain_fast_planner.control.delayed_vlm_score_fusion import (
    FusionConfig,
    PlannerScoreModel,
    SimLoopConfig,
    ToyTask,
    VlmDelayModel,
    load_static_candidate_set,
    make_task,
    run_toy_simulation,
    run_toy_simulation_with_trace,
)
from slow_brain_fast_planner.control.tracking import (
    EndpointPDConfig,
    Limits,
    PurePursuitConfig,
    project_point_to_polyline_arclength,
)

# --- Choose a trajectory library ---
CAND_JSON = (
    REPO
    / "assets/trajectory_selection_static_candidates/takeover_kmeans_medoids"
    / "static_candidates_k12.json"
)

candidates_body = load_static_candidate_set(CAND_JSON, prepend_origin=True)
print("candidates:", len(candidates_body), "points each:", candidates_body[0].shape[0])

In [ ]:
def plot_candidate_library(
    candidates: list[np.ndarray],
    *,
    title: str = "candidate library (robot frame)",
    max_show: int | None = None,
) -> None:
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    max_show = len(candidates) if max_show is None else int(max_show)
    for i, c in enumerate(candidates[:max_show]):
        c = np.asarray(c, dtype=np.float64)
        ax.plot(c[:, 1], c[:, 0], lw=1.8, alpha=0.8)
        ax.text(float(c[-1, 1]), float(c[-1, 0]), str(i), fontsize=9, alpha=0.9)

    ax.scatter([0.0], [0.0], c="k", s=30)
    ax.arrow(0.0, 0.0, 0.0, 0.6, head_width=0.12, head_length=0.16, fc="k", ec="k")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.set_xlabel("y_left (m)")
    ax.set_ylabel("x_forward (m)")
    ax.set_title(title)
    plt.show()


plot_candidate_library(candidates_body)

In [ ]:
from slow_brain_fast_planner.control.tracking import SimConfig, simulate_tracking


def _load_navflow_robot_limits() -> tuple[float, float]:
    """Load (max_v, max_w) from NavFlow's deploy config if present.

    Avoids adding a hard dependency on PyYAML for the notebook.
    """
    p = REPO / "deploy/NavFlow/visualnav_ros/deployment/config/robot.yaml"
    if not p.exists():
        return 1.0, 0.5
    txt = p.read_text(encoding="utf-8")

    # Try PyYAML first (if installed).
    try:
        import yaml  # type: ignore

        cfg = yaml.safe_load(txt)
        if isinstance(cfg, dict):
            return float(cfg.get("max_v", 1.0)), float(cfg.get("max_w", 0.5))
    except Exception:
        pass

    # Fallback: parse simple `key: value` lines.
    mv = None
    mw = None
    for line in txt.splitlines():
        line = line.split("#", 1)[0].strip()
        if not line or ":" not in line:
            continue
        k, v = line.split(":", 1)
        k = k.strip()
        v = v.strip()
        try:
            if k == "max_v":
                mv = float(v)
            elif k == "max_w":
                mw = float(v)
        except Exception:
            continue
    return float(mv if mv is not None else 1.0), float(mw if mw is not None else 0.5)


MAX_V, MAX_W = _load_navflow_robot_limits()
print("robot.yaml limits:", {"max_v": MAX_V, "max_w": MAX_W})

# --- Limits used by the tiny simulator ---
# Defaults match `deploy/NavFlow/visualnav_ros/deployment/src/pure_pursuit_controller_ros2.py`.
limits = Limits(
    max_v=float(MAX_V),
    max_w=float(MAX_W),
    max_a=1.0,
    max_alpha=1.2,
    max_lat_accel=0.8,
)

# --- Controller knobs ---
pp_cfg = PurePursuitConfig(lookahead_m=1.0, lookahead_gain=0.5, v_gain=0.8)
# Match latest on-robot PD-style settings (no curvature slow-down by default).
ep_cfg = EndpointPDConfig(dt_nominal=1.0, curvature_speed_gain=0.0)

# --- Simulation timing for tracking-only sanity check ---
tracking_sim = SimConfig(dt=0.1, horizon_s=4.0, limits=limits)

In [ ]:
def ideal_body_path(
    *, v_ref: float, w_ref: float, horizon_s: float, n_points: int, prepend_origin: bool
) -> np.ndarray:
    """Ideal commanded short-horizon path in robot frame.

    This matches the toy sim's intended ideal: constant v, constant w (unicycle),
    starting at origin.
    """
    n = int(n_points)
    if n < 2:
        raise ValueError("n_points must be >= 2")
    if prepend_origin:
        t = np.linspace(0.0, float(horizon_s), n, dtype=np.float64)
    else:
        t = np.linspace(float(horizon_s) / float(n), float(horizon_s), n, dtype=np.float64)

    v = float(v_ref)
    w = float(w_ref)
    if abs(w) < 1e-9:
        x = v * t
        y = np.zeros_like(x)
        return np.stack([x, y], axis=1)

    k = w / max(1e-9, v)  # curvature rad/m
    s = v * t
    x = np.sin(k * s) / k
    y = (1.0 - np.cos(k * s)) / k
    return np.stack([x, y], axis=1)


def summarize_tracking(ref_body_xy: np.ndarray) -> dict:
    ep = simulate_tracking(
        ref_body_xy, controller="endpoint_pd", sim=tracking_sim, pp_cfg=pp_cfg, ep_cfg=ep_cfg
    )
    pp = simulate_tracking(
        ref_body_xy, controller="pure_pursuit", sim=tracking_sim, pp_cfg=pp_cfg, ep_cfg=ep_cfg
    )
    return {
        "endpoint_pd": {k: ep[k] for k in ["cte_mean", "cte_p95", "end_err", "progress_frac"]},
        "pure_pursuit": {k: pp[k] for k in ["cte_mean", "cte_p95", "end_err", "progress_frac"]},
    }

In [ ]:
# --- Controller sanity check on idealized tasks (no VLM, no candidate selection) ---

for task_name in ["forward", "left_turn", "right_turn"]:
    task = make_task(task_name, v_ref=1.3, radius_m=2.5)
    ref = ideal_body_path(
        v_ref=task.v_ref,
        w_ref=task.w_ref,
        horizon_s=4.0,
        n_points=int(candidates_body[0].shape[0]),
        prepend_origin=True,
    )
    stats = summarize_tracking(ref)
    print(task_name, stats)

    fig, ax = plt.subplots(figsize=(6.6, 5.8))
    ax.plot(ref[:, 1], ref[:, 0], lw=3, color="0.35")
    ax.scatter([0.0], [0.0], c="k", s=30)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.set_xlabel("y_left (m)")
    ax.set_ylabel("x_forward (m)")
    ax.set_title(
        f"ideal reference path: {task_name}  (v={task.v_ref:.2f} m/s, w={task.w_ref:.2f} rad/s)"
    )
    plt.show()

In [ ]:
# --- Toy sim config (delayed VLM + score fusion) ---

sim_cfg = SimLoopConfig(
    dt_control=0.1,
    dt_plan=0.2,
    horizon_s=4.0,
    prepend_origin=True,
    limits=limits,
    pp_cfg=pp_cfg,
    ep_cfg=ep_cfg,
)

score_model = PlannerScoreModel(
    noise_std=0.15,  # synthetic planner score noise
    score_scale=1.0,
)

fusion_cfg = FusionConfig(
    enabled=True,
    similarity_mode="body_pointwise_arclen",  # usually most stable when stale plan is from old pose
    lambda_sim=1.0,
    staleness_tau_s=3.0,
    dist_scale_m=1.0,
    align_to_current_pose=True,
)

In [ ]:
def compute_trace_metrics(trace) -> dict[str, np.ndarray]:
    # pointwise errors vs ideal world trajectory
    err = np.linalg.norm(trace.actual_world - trace.ideal_world, axis=1)

    # cross-track error to the ideal world polyline (slower but fine for toy sizes)
    cte = np.zeros_like(err)
    for i in range(trace.actual_world.shape[0]):
        _s, d = project_point_to_polyline_arclength(trace.actual_world[i], trace.ideal_world)
        cte[i] = float(d)

    return {
        "pos_err_m": err,
        "cte_m": cte,
    }


def plot_single_run(
    trace,
    *,
    title: str = "",
    show_selected_plans: bool = True,
    show_vlm_plans: bool = False,
    max_plan_overlays: int = 8,
) -> None:
    """Plot one simulation run.

    Color code (bird's-eye):
    - gray: ideal (GT) world trajectory
    - blue: executed robot trajectory
    - orange: selected plan(s) at planner ticks (what the controller is tracking between replans)
    - green: last VLM plan at planner ticks (stale / delayed), shown only if enabled
    """
    m = compute_trace_metrics(trace)

    from matplotlib.lines import Line2D

    C_IDEAL = "0.35"
    C_EXEC = "#1f77b4"
    C_SEL = "#ff7f0e"
    C_VLM = "#2ca02c"

    # Bird's-eye view
    fig, ax = plt.subplots(figsize=(15, 12))
    ax.plot(trace.ideal_world[:, 1], trace.ideal_world[:, 0], color=C_IDEAL, lw=3.2)
    ax.plot(trace.actual_world[:, 1], trace.actual_world[:, 0], color=C_EXEC, lw=2.8)

    # Overlay selected plans (these ARE the selected trajectories)
    if show_selected_plans and len(trace.ref_world) > 0:
        step = max(1, int(len(trace.ref_world) // max(1, int(max_plan_overlays))))
        for i in range(0, len(trace.ref_world), step):
            p = np.asarray(trace.ref_world[i], dtype=np.float64)
            ax.plot(p[:, 1], p[:, 0], color=C_SEL, lw=1.3, alpha=0.25)
        # Highlight the last plan more strongly.
        plast = np.asarray(trace.ref_world[-1], dtype=np.float64)
        ax.plot(plast[:, 1], plast[:, 0], color=C_SEL, lw=2.8, alpha=0.9)

    # Overlay the VLM plan (stale) if requested
    if show_vlm_plans and len(trace.vlm_ref_world) > 0:
        step = max(1, int(len(trace.vlm_ref_world) // max(1, int(max_plan_overlays))))
        for i in range(0, len(trace.vlm_ref_world), step):
            if trace.vlm_ref_world[i] is None:
                continue
            p = np.asarray(trace.vlm_ref_world[i], dtype=np.float64)
            ax.plot(p[:, 1], p[:, 0], color=C_VLM, lw=1.2, alpha=0.20)
        # Highlight the last available VLM plan.
        for j in range(len(trace.vlm_ref_world) - 1, -1, -1):
            if trace.vlm_ref_world[j] is None:
                continue
            p = np.asarray(trace.vlm_ref_world[j], dtype=np.float64)
            ax.plot(p[:, 1], p[:, 0], color=C_VLM, lw=2.4, alpha=0.8)
            break

    ax.scatter([0.0], [0.0], c="k", s=30)
    ax.arrow(0.0, 0.0, 0.0, 0.6, head_width=0.12, head_length=0.16, fc="k", ec="k")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.set_xlabel("y_left (m)")
    ax.set_ylabel("x_forward (m)")
    ax.set_title(title or "bird's-eye")

    # Explicit legend (so orange/green are not ambiguous)
    handles = [
        Line2D([0], [0], color=C_IDEAL, lw=3.2, label="ideal (GT)"),
        Line2D([0], [0], color=C_EXEC, lw=2.8, label="executed"),
    ]
    if show_selected_plans:
        handles.append(
            Line2D([0], [0], color=C_SEL, lw=2.8, label="selected trajectory (plan tick)")
        )
    if show_vlm_plans:
        handles.append(
            Line2D([0], [0], color=C_VLM, lw=2.4, label="VLM trajectory (stale / delayed)")
        )
    ax.legend(handles=handles, loc="best")

    plt.show()

    # Time series
    fig, axs = plt.subplots(4, 1, figsize=(9.2, 10.6), sharex=True)
    axs[0].plot(trace.t_s, m["pos_err_m"], lw=2)
    axs[0].set_ylabel("pos err (m)")
    axs[0].grid(True, alpha=0.25)

    axs[1].plot(trace.t_s, m["cte_m"], lw=2)
    axs[1].set_ylabel("CTE to ideal (m)")
    axs[1].grid(True, alpha=0.25)

    axs[2].plot(trace.t_s, trace.v, lw=2, label="v")
    axs[2].axhline(float(limits.max_v), color="0.4", lw=1, ls="--", alpha=0.7)
    axs[2].set_ylabel("v (m/s)")
    axs[2].grid(True, alpha=0.25)

    axs[3].plot(trace.t_s, trace.w, lw=2, label="w")
    axs[3].axhline(float(limits.max_w), color="0.4", lw=1, ls="--", alpha=0.7)
    axs[3].axhline(-float(limits.max_w), color="0.4", lw=1, ls="--", alpha=0.7)
    axs[3].set_ylabel("w (rad/s)")
    axs[3].set_xlabel("time (s)")
    axs[3].grid(True, alpha=0.25)

    fig.suptitle(title or "time series")
    fig.tight_layout()
    plt.show()

    # Planner-tick diagnostics
    if trace.plan_t_s.size > 0:
        fig, axs = plt.subplots(3, 1, figsize=(9.2, 7.8), sharex=True)
        axs[0].step(trace.plan_t_s, trace.chosen_idx, where="post")
        axs[0].set_ylabel("chosen idx")
        axs[0].grid(True, alpha=0.25)

        axs[1].plot(trace.plan_t_s, trace.staleness_s, lw=2)
        axs[1].set_ylabel("staleness (s)")
        axs[1].grid(True, alpha=0.25)

        axs[2].plot(trace.plan_t_s, trace.fusion_weight, lw=2)
        axs[2].set_ylabel("fusion weight")
        axs[2].set_xlabel("plan time (s)")
        axs[2].grid(True, alpha=0.25)

        fig.suptitle((title + "  | plan-tick") if title else "plan-tick")
        fig.tight_layout()
        plt.show()

In [ ]:
# --- Single run (3 tasks at once): forward + left + right ---
# This cell is designed to sanity-check:
#   (1) candidate library quality
#   (2) controller tracking behavior
#   (3) delayed VLM logic
# across *the same* robot limits and timing.

from matplotlib.lines import Line2D

CONTROLLER = "endpoint_pd"  # "endpoint_pd" | "pure_pursuit"
SELECTION_MODE = "score_fusion"  # "oracle_min_ade" | "noisy_planner" | "score_fusion"
VLM_DELAY_S = 1.5
SEED = 0

V_REF = 0.8
LEFT_RADIUS_M = 4.0
RIGHT_RADIUS_M = 8.0
DURATION_S = 20.0

# Candidate library options:
# - "assets_kmeans12": current default asset set
# - "assets_kmeans12": paper-default k-means medoid set
# - "grid64": generated on-the-fly (8x8 endpoints)
CAND_MODE = "assets_kmeans12"


def make_grid64_candidate_set(
    *,
    x_range_m: tuple[float, float] = (0.8, 5.0),
    y_range_m: tuple[float, float] = (-3.0, 3.0),
    nx: int = 8,
    ny: int = 8,
    n_points: int = 21,
    prepend_origin: bool = True,
) -> list[np.ndarray]:
    """Generate a simple grid of (x_end, y_end) endpoints and connect with constant-curvature arcs.

    This is a *geometry* library (no dynamics). Points are in robot frame (x forward, y left).
    """

    def _arc_to_endpoint(x_end: float, y_end: float) -> np.ndarray:
        x_end = float(x_end)
        y_end = float(y_end)
        n = int(n_points)
        if n < 2:
            raise ValueError("n_points must be >=2")

        # Straight line (y ~ 0)
        if abs(y_end) < 1e-6:
            xs = np.linspace(0.0, x_end, n, dtype=np.float64)
            ys = np.zeros_like(xs)
            pts = np.stack([xs, ys], axis=1)
        else:
            # Use circular-arc endpoint parameterization:
            # x = R sin(theta), y = R(1 - cos(theta)), so tan(theta/2) = y/x.
            theta = 2.0 * float(np.arctan2(y_end, max(1e-6, x_end)))
            # Avoid degenerate
            if abs(theta) < 1e-6:
                xs = np.linspace(0.0, x_end, n, dtype=np.float64)
                ys = np.zeros_like(xs)
                pts = np.stack([xs, ys], axis=1)
            else:
                R = x_end / float(np.sin(theta))
                u = np.linspace(0.0, 1.0, n, dtype=np.float64)
                th = u * theta
                xs = R * np.sin(th)
                ys = R * (1.0 - np.cos(th))
                pts = np.stack([xs, ys], axis=1)

        if prepend_origin:
            pts0 = np.zeros((pts.shape[0] + 1, 2), dtype=np.float64)
            pts0[1:, :] = pts
            pts = pts0
        return np.asarray(pts, dtype=np.float64)

    xs = np.linspace(float(x_range_m[0]), float(x_range_m[1]), int(nx), dtype=np.float64)
    ys = np.linspace(float(y_range_m[0]), float(y_range_m[1]), int(ny), dtype=np.float64)

    out: list[np.ndarray] = []
    for x in xs:
        for y in ys:
            # Skip endpoints behind the robot.
            if float(x) <= 0.05:
                continue
            out.append(_arc_to_endpoint(float(x), float(y)))

    # De-dup any NaNs / infs
    out2 = []
    for p in out:
        if p.ndim == 2 and p.shape[1] == 2 and np.all(np.isfinite(p)):
            out2.append(p)
    return out2


# Select candidate set
if CAND_MODE == "assets_kmeans12":
    cand_path = (
        REPO
        / "assets/trajectory_selection_static_candidates/takeover_kmeans_medoids"
        / "static_candidates_k12.json"
    )
    candidates = load_static_candidate_set(cand_path, prepend_origin=True)
elif CAND_MODE == "grid64":
    candidates = make_grid64_candidate_set(n_points=21, prepend_origin=True)
else:
    candidates = candidates_body

print("candidate_set:", CAND_MODE, "K=", len(candidates), "points=", candidates[0].shape[0])


# Selection policy mapping
if SELECTION_MODE == "oracle_min_ade":
    # Oracle selection each plan tick: delay=0 and hold the oracle-chosen plan.
    run_policy = "vlm_hold"
    run_delay = 0.0
elif SELECTION_MODE == "score_fusion":
    run_policy = "score_fusion"
    run_delay = float(VLM_DELAY_S)
else:
    run_policy = "local_only"  # noisy planner-score argmax baseline
    run_delay = float(VLM_DELAY_S)


def run_one(task_name: str, *, radius_m: float | None) -> tuple[dict, object, object]:
    if task_name == "forward":
        task0 = make_task("forward", v_ref=float(V_REF), radius_m=float(LEFT_RADIUS_M))
    else:
        task0 = make_task(task_name, v_ref=float(V_REF), radius_m=float(radius_m))

    task = ToyTask(
        name=task0.name, v_ref=task0.v_ref, w_ref=task0.w_ref, duration_s=float(DURATION_S)
    )
    vlm_cfg = VlmDelayModel(delay_s=float(run_delay), oracle=True)

    res, trace = run_toy_simulation_with_trace(
        candidates_body=candidates,
        task=task,
        controller=CONTROLLER,
        policy=run_policy,
        sim_cfg=sim_cfg,
        score_model=score_model,
        vlm_cfg=vlm_cfg,
        fusion_cfg=fusion_cfg,
        seed=int(SEED),
    )

    switches = int(np.count_nonzero(np.diff(np.asarray(res.chosen_idx_hist, dtype=np.int64))))
    summary = {"task": task_name, **asdict(res), "chosen_switches": switches}
    return summary, res, trace


summaries = {}
traces = {}
for name, rad in [
    ("forward", None),
    ("left_turn", float(LEFT_RADIUS_M)),
    ("right_turn", float(RIGHT_RADIUS_M)),
]:
    s, _res, tr = run_one(name, radius_m=rad)
    summaries[name] = s
    traces[name] = tr

# Print quick metrics table
if pd is not None:
    display(pd.DataFrame([summaries[k] for k in ["forward", "left_turn", "right_turn"]]))
else:
    for k in ["forward", "left_turn", "right_turn"]:
        print(
            k,
            summaries[k]["mean_pos_err_m"],
            summaries[k]["mean_cte_to_ideal_m"],
            summaries[k]["chosen_switches"],
        )


# --- Plot 1: bird's-eye (3 tasks together) ---
COL = {"forward": "#1f77b4", "left_turn": "#ff7f0e", "right_turn": "#2ca02c"}
MRK = {"forward": "o", "left_turn": "s", "right_turn": "^"}
LBL = {
    "forward": "forward",
    "left_turn": f"left (R={LEFT_RADIUS_M:g}m)",
    "right_turn": f"right (R={RIGHT_RADIUS_M:g}m)",
}

fig, ax = plt.subplots(figsize=(8.8, 7.1))
for k in ["forward", "left_turn", "right_turn"]:
    tr = traces[k]
    c = COL[k]

    # Ideal and executed
    ax.plot(
        tr.ideal_world[:, 1], tr.ideal_world[:, 0], ls="--", lw=2.2, color=c, alpha=0.70, zorder=2
    )
    ax.plot(tr.actual_world[:, 1], tr.actual_world[:, 0], lw=3.2, color=c, alpha=0.95, zorder=3)
    # End marker (helps verify e.g. forward executed exists even if it overlaps the ideal)
    ax.scatter(
        [tr.actual_world[-1, 1]],
        [tr.actual_world[-1, 0]],
        s=70,
        marker="x",
        color=c,
        linewidths=2.5,
        zorder=6,
    )

    # Start marker (hollow + thick edge so all 3 markers are visible even though they
    # overlap at (0,0))
    ax.scatter(
        [tr.actual_world[0, 1]],
        [tr.actual_world[0, 0]],
        s=190,
        marker=MRK[k],
        facecolors="none",
        edgecolors=c,
        linewidths=3.0,
        zorder=9,
    )
    # Label each start marker with different text offsets (so F/L/R don't overlap)
    text_off = {
        "forward": (0.20, 0.55),
        "left_turn": (0.20, 0.25),
        "right_turn": (0.20, 0.85),
    }[k]
    ax.text(
        float(tr.actual_world[0, 1]) + float(text_off[0]),
        float(tr.actual_world[0, 0]) + float(text_off[1]),
        k[0].upper(),
        fontsize=12,
        weight="bold",
        color=c,
        zorder=10,
    )

    # Also show the *last* selected plan (dotted) to verify selection/controller alignment.
    if len(tr.ref_world) > 0:
        plast = np.asarray(tr.ref_world[-1], dtype=np.float64)
        ax.plot(plast[:, 1], plast[:, 0], ls=":", lw=2.4, color=c, alpha=0.75)

ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)
ax.set_xlabel("y_left (m)")
ax.set_ylabel("x_forward (m)")
ax.set_title(
    f"3-task overlay | ctrl={CONTROLLER} | selector={SELECTION_MODE} "
    f"| policy={run_policy} | delay={run_delay:g}s | cand={CAND_MODE}"
)

handles = []
for k in ["forward", "left_turn", "right_turn"]:
    handles.append(Line2D([0], [0], color=COL[k], lw=3.0, label=f"{LBL[k]} executed"))
    handles.append(Line2D([0], [0], color=COL[k], lw=2.2, ls="--", label=f"{LBL[k]} ideal"))
handles.append(Line2D([0], [0], color="k", lw=2.4, ls=":", label="(dotted) last selected plan"))
ax.legend(handles=handles, loc="best", fontsize=9)
plt.show()


# --- Plot 2: metrics + state timeline (3 tasks together) ---
fig, axs = plt.subplots(4, 1, figsize=(10.0, 10.8), sharex=True)
for k in ["forward", "left_turn", "right_turn"]:
    tr = traces[k]
    m = compute_trace_metrics(tr)
    c = COL[k]
    label = LBL[k]
    axs[0].plot(tr.t_s, m["pos_err_m"], color=c, lw=2.1, label=label)
    axs[1].plot(tr.t_s, m["cte_m"], color=c, lw=2.1, label=label)
    axs[2].plot(tr.t_s, tr.v, color=c, lw=2.1, label=label)
    axs[3].plot(tr.t_s, tr.w, color=c, lw=2.1, label=label)

axs[0].set_ylabel("pos err (m)")
axs[1].set_ylabel("CTE (m)")
axs[2].set_ylabel("v (m/s)")
axs[3].set_ylabel("w (rad/s)")
axs[3].set_xlabel("time (s)")
for ax_ in axs:
    ax_.grid(True, alpha=0.25)
axs[0].legend(loc="upper right")
fig.suptitle(
    f"metrics/state overlay | ctrl={CONTROLLER} | selector={SELECTION_MODE} | cand={CAND_MODE}"
)
fig.tight_layout()
plt.show()


# Notes:
# - If `endpoint_pd` looks like it "drives away" on turns, it is usually because
#   endpoint-based w is too small
#   (it tends to under-steer, producing a larger-radius path). Compare with
#   `pure_pursuit` to validate.
# - `oracle_min_ade` here is implemented as `vlm_hold` with delay=0
#   (oracle selection each plan tick).

In [ ]:
def _normalize_task_specs(
    *,
    task_specs: list[dict] | None,
    tasks: tuple[str, ...],
    v_ref: float,
    duration_s: float,
    radius_m: float,
    left_radii_m: tuple[float, ...],
    right_radii_m: tuple[float, ...],
) -> list[dict]:
    """Build a normalized list of task specs.

    Supported inputs:
    - **task_specs**: explicit list of dict specs.
      Each spec may contain: `task` (required), `tag`, `v_ref`, `radius_m`, `duration_s`.

    - **tasks + radius params** (backward compatible):
      - tasks may include: "forward", "left_turn", "right_turn"
      - `radius_m` is used for turns by default.
      - Use `left_radii_m`/`right_radii_m` to make multiple variants.

    Output format:
    - dict with keys: task, tag, v_ref, radius_m, duration_s
    """
    if task_specs is not None:
        out = []
        for s in task_specs:
            if not isinstance(s, dict) or "task" not in s:
                raise ValueError("Each task spec must be a dict with key 'task'.")
            out.append(
                {
                    "task": str(s["task"]),
                    "tag": str(s.get("tag", s["task"])),
                    "v_ref": float(s.get("v_ref", v_ref)),
                    "radius_m": None
                    if s.get("radius_m", None) is None
                    else float(s.get("radius_m")),
                    "duration_s": float(s.get("duration_s", duration_s)),
                }
            )
        return out

    # Backward-compatible expansion
    out: list[dict] = []
    for t in tasks:
        t = str(t)
        if t == "forward":
            out.append(
                {
                    "task": "forward",
                    "tag": "forward",
                    "v_ref": float(v_ref),
                    "radius_m": None,
                    "duration_s": float(duration_s),
                }
            )
        elif t == "left_turn":
            radii = left_radii_m if left_radii_m else (float(radius_m),)
            for r in radii:
                out.append(
                    {
                        "task": "left_turn",
                        "tag": f"left_turn_R{float(r):g}",
                        "v_ref": float(v_ref),
                        "radius_m": float(r),
                        "duration_s": float(duration_s),
                    }
                )
        elif t == "right_turn":
            radii = right_radii_m if right_radii_m else (float(radius_m),)
            for r in radii:
                out.append(
                    {
                        "task": "right_turn",
                        "tag": f"right_turn_R{float(r):g}",
                        "v_ref": float(v_ref),
                        "radius_m": float(r),
                        "duration_s": float(duration_s),
                    }
                )
        else:
            raise ValueError(f"Unknown task name: {t}")
    return out


def run_sweep(
    *,
    # New: explicit specs (best for diverse tasks)
    task_specs: list[dict] | None = None,
    # Backward compatible
    tasks=("forward", "left_turn", "right_turn"),
    v_ref: float = 1.3,
    duration_s: float = 12.0,
    radius_m: float = 2.5,
    left_radii_m: tuple[float, ...] = (),
    right_radii_m: tuple[float, ...] = (),
    # Sweep axes
    controllers=("endpoint_pd", "pure_pursuit"),
    policies=("local_only", "vlm_hold", "score_fusion"),
    delays_s=(0.0, 0.5, 1.5, 2.0),
    seeds=(0, 1, 2, 3, 4),
    # Other knobs
    candidates: list[np.ndarray] | None = None,
    fusion_cfg_override: FusionConfig | None = None,
) -> list[dict]:
    """Run a sweep across tasks/controllers/policies/delays.

    For *left/center/right* style tasks, pass e.g.:

    task_specs=[
      {"task":"forward", "tag":"center"},
      {"task":"left_turn", "radius_m":4.0, "tag":"left_R4"},
      {"task":"right_turn", "radius_m":8.0, "tag":"right_R8"},
    ]
    """

    cand = candidates_body if candidates is None else candidates
    specs = _normalize_task_specs(
        task_specs=task_specs,
        tasks=tuple(tasks),
        v_ref=float(v_ref),
        duration_s=float(duration_s),
        radius_m=float(radius_m),
        left_radii_m=tuple(left_radii_m),
        right_radii_m=tuple(right_radii_m),
    )

    # tqdm progress (optional)
    try:
        from tqdm.auto import tqdm  # type: ignore
    except Exception:  # noqa: BLE001
        tqdm = None

    total = (
        int(len(specs))
        * int(len(tuple(controllers)))
        * int(len(tuple(policies)))
        * int(len(tuple(delays_s)))
        * int(len(tuple(seeds)))
    )
    pbar = tqdm(total=total, desc="sweep", unit="run") if tqdm is not None else None

    rows: list[dict] = []
    try:
        for spec in specs:
            task_name = str(spec["task"])
            tag = str(spec["tag"])
            this_v = float(spec["v_ref"])
            this_dur = float(spec["duration_s"])
            this_r = spec.get("radius_m", None)

            # make_task ignores radius_m for forward
            task0 = make_task(
                task_name,
                v_ref=this_v,
                radius_m=float(this_r) if this_r is not None else float(radius_m),
            )
            task = ToyTask(
                name=task0.name, v_ref=task0.v_ref, w_ref=task0.w_ref, duration_s=float(this_dur)
            )

            for controller in controllers:
                for policy in policies:
                    for delay_s in delays_s:
                        vlm_cfg = VlmDelayModel(delay_s=float(delay_s), oracle=True)
                        for seed in seeds:
                            res = run_toy_simulation(
                                candidates_body=cand,
                                task=task,
                                controller=controller,
                                policy=policy,
                                sim_cfg=sim_cfg,
                                score_model=score_model,
                                vlm_cfg=vlm_cfg,
                                fusion_cfg=(fusion_cfg_override or fusion_cfg),
                                seed=int(seed),
                            )
                            chosen = np.asarray(res.chosen_idx_hist, dtype=np.int64)
                            switches = (
                                int(np.count_nonzero(np.diff(chosen))) if chosen.size >= 2 else 0
                            )
                            rows.append(
                                {
                                    # task id (unique for plots)
                                    "task": tag,
                                    "task_type": str(task_name),
                                    "radius_m": float(this_r)
                                    if this_r is not None
                                    else float("nan"),
                                    "v_ref": float(this_v),
                                    "duration_s": float(this_dur),
                                    "controller": str(controller),
                                    "policy": str(policy),
                                    "delay_s": float(delay_s),
                                    "seed": int(seed),
                                    "mean_pos_err_m": float(res.mean_pos_err_m),
                                    "p95_pos_err_m": float(res.p95_pos_err_m),
                                    "mean_cte_to_ideal_m": float(res.mean_cte_to_ideal_m),
                                    "p95_cte_to_ideal_m": float(res.p95_cte_to_ideal_m),
                                    "mean_speed_mps": float(res.mean_speed_mps),
                                    "stopped_frac": float(res.stopped_frac),
                                    "mean_abs_dv": float(res.mean_abs_dv),
                                    "mean_abs_dw": float(res.mean_abs_dw),
                                    "vlm_updates": int(res.vlm_updates),
                                    "chosen_switches": int(switches),
                                }
                            )
                            if pbar is not None:
                                pbar.update(1)
                                # lightweight live context
                                pbar.set_postfix(
                                    {
                                        "task": tag,
                                        "ctrl": str(controller),
                                        "pol": str(policy),
                                        "delay": float(delay_s),
                                        "seed": int(seed),
                                    }
                                )
    finally:
        if pbar is not None:
            pbar.close()

    return rows


def to_frame(rows: list[dict]):
    if pd is None:
        return rows
    df = pd.DataFrame(rows)
    return df

In [ ]:
# --- Run a sweep (can take ~seconds to a minute depending on grid size) ---

rows = run_sweep(
    task_specs=[
        {"task": "forward", "tag": "center"},
        {"task": "left_turn", "radius_m": 4.0, "tag": "left_R4"},
        {"task": "right_turn", "radius_m": 8.0, "tag": "right_R8"},
    ],
    controllers=("endpoint_pd", "pure_pursuit"),
    policies=("local_only", "vlm_hold", "score_fusion"),
    delays_s=(0.0, 0.5, 1.5, 2.0),
    seeds=(0,),
)

df = to_frame(rows)

if pd is None:
    print("pandas not available; got rows:", len(rows))
    print(rows[0])
else:
    display(df.groupby(["controller", "policy", "delay_s"]).mean(numeric_only=True))
    # display(df.head())
    # display(df.groupby(["task", "controller", "policy", "delay_s"])
    #         .mean(numeric_only=True).reset_index().head())

In [ ]:
display(df.groupby(["controller", "policy", "delay_s"]).mean(numeric_only=True))

In [ ]:
raise ValueError("stop")


def heatmap(df, *, metric: str, task: str, controller: str) -> None:
    if pd is None:
        raise RuntimeError("heatmap requires pandas")

    sub = df[(df.task == task) & (df.controller == controller)].copy()
    agg = sub.groupby(["policy", "delay_s"])[metric].mean().reset_index()

    # rows=policy, cols=delay
    piv = agg.pivot(index="policy", columns="delay_s", values=metric)
    policies = list(piv.index)
    delays = list(piv.columns)
    Z = piv.to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(7.6, 2.6 + 0.45 * len(policies)))
    im = ax.imshow(Z, aspect="auto")

    ax.set_xticks(range(len(delays)))
    ax.set_xticklabels([f"{d:.1f}" for d in delays])
    ax.set_yticks(range(len(policies)))
    ax.set_yticklabels(policies)

    ax.set_xlabel("VLM delay (s)")
    ax.set_ylabel("policy")
    ax.set_title(f"{metric}  | task={task}  ctrl={controller}")

    # annotate
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]):
            v = Z[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.3g}", ha="center", va="center", fontsize=10, color="w")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()


if pd is not None:
    for task in ["forward", "left_turn", "right_turn"]:
        for ctrl in ["endpoint_pd", "pure_pursuit"]:
            heatmap(df, metric="mean_pos_err_m", task=task, controller=ctrl)
            heatmap(df, metric="mean_cte_to_ideal_m", task=task, controller=ctrl)
            heatmap(df, metric="chosen_switches", task=task, controller=ctrl)

In [ ]:
# --- Optional: fusion ablation sweep (lambda_sim, tau, similarity mode) ---
# Keep this small at first; increase grid size once plots look sensible.


raise ValueError("stop")

if pd is not None:
    ab_rows: list[dict] = []
    for sim_mode in ["body_pointwise_arclen", "pointwise_arclen", "candidate_to_ref_polyline"]:
        for lam in [0.0, 0.5, 1.0, 2.0]:
            for tau in [1.0, 2.0, 3.0, 6.0]:
                fcfg = FusionConfig(
                    enabled=True,
                    similarity_mode=sim_mode,
                    lambda_sim=float(lam),
                    staleness_tau_s=float(tau),
                    dist_scale_m=1.0,
                    align_to_current_pose=True,
                )
                r = run_sweep(
                    tasks=("left_turn",),
                    controllers=("pure_pursuit",),
                    policies=("score_fusion",),
                    delays_s=(1.5, 2.0),
                    seeds=(0, 1, 2, 3, 4),
                    fusion_cfg_override=fcfg,
                )
                for row in r:
                    row["similarity_mode"] = sim_mode
                    row["lambda_sim"] = float(lam)
                    row["staleness_tau_s"] = float(tau)
                ab_rows.extend(r)

    ab = pd.DataFrame(ab_rows)
    display(
        ab.groupby(["similarity_mode", "lambda_sim", "staleness_tau_s", "delay_s"])
        .mean(numeric_only=True)
        .reset_index()
        .sort_values(["delay_s", "mean_pos_err_m"])
        .head(20)
    )

In [ ]:
raise ValueError("stop")


def compare_policies(
    *,
    task_name: str,
    controller: str,
    delay_s: float,
    seed: int = 0,
    policies=("local_only", "vlm_hold", "score_fusion"),
) -> None:
    task0 = make_task(task_name, v_ref=1.3, radius_m=2.5)
    task = ToyTask(name=task0.name, v_ref=task0.v_ref, w_ref=task0.w_ref, duration_s=12.0)
    vlm_cfg = VlmDelayModel(delay_s=float(delay_s), oracle=True)

    outs = []
    for pol in policies:
        res, trace = run_toy_simulation_with_trace(
            candidates_body=candidates_body,
            task=task,
            controller=controller,
            policy=pol,
            sim_cfg=sim_cfg,
            score_model=score_model,
            vlm_cfg=vlm_cfg,
            fusion_cfg=fusion_cfg,
            seed=int(seed),
        )
        switches = int(np.count_nonzero(np.diff(np.asarray(res.chosen_idx_hist, dtype=np.int64))))
        outs.append((pol, res, trace, switches))

    # Bird's-eye side-by-side
    from matplotlib.lines import Line2D

    C_IDEAL = "0.35"
    C_EXEC = "#1f77b4"
    C_SEL = "#ff7f0e"

    fig, axs = plt.subplots(
        1, len(outs), figsize=(6.8 * len(outs), 6.1), sharex=False, sharey=False
    )
    if len(outs) == 1:
        axs = [axs]

    for ax, (pol, res, trace, switches) in zip(axs, outs, strict=False):
        ax.plot(trace.ideal_world[:, 1], trace.ideal_world[:, 0], color=C_IDEAL, lw=3.2)
        ax.plot(trace.actual_world[:, 1], trace.actual_world[:, 0], color=C_EXEC, lw=2.8)
        if len(trace.ref_world) > 0:
            step = max(1, len(trace.ref_world) // 6)
            for i in range(0, len(trace.ref_world), step):
                p = np.asarray(trace.ref_world[i], dtype=np.float64)
                ax.plot(p[:, 1], p[:, 0], color=C_SEL, lw=1.2, alpha=0.20)
            plast = np.asarray(trace.ref_world[-1], dtype=np.float64)
            ax.plot(plast[:, 1], plast[:, 0], color=C_SEL, lw=2.6, alpha=0.85)

        ax.scatter([0.0], [0.0], c="k", s=30)
        ax.arrow(0.0, 0.0, 0.0, 0.6, head_width=0.12, head_length=0.16, fc="k", ec="k")
        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, alpha=0.25)
        ax.set_xlabel("y_left (m)")
        ax.set_ylabel("x_forward (m)")
        ax.set_title(
            f"{pol}\nmean_pos_err={res.mean_pos_err_m:.3f}  "
            f"mean_cte={res.mean_cte_to_ideal_m:.3f}\n"
            f"mean_v={res.mean_speed_mps:.2f}  stopped={res.stopped_frac:.2f}  switches={switches}"
        )

    # Shared legend
    handles = [
        Line2D([0], [0], color=C_IDEAL, lw=3.2, label="ideal (GT)"),
        Line2D([0], [0], color=C_EXEC, lw=2.8, label="executed"),
        Line2D([0], [0], color=C_SEL, lw=2.6, label="selected trajectory (plan tick)"),
    ]
    fig.legend(handles=handles, loc="upper center", ncol=3)

    fig.suptitle(
        f"compare policies | task={task_name}  ctrl={controller}  delay={delay_s:.1f}s  seed={seed}"
    )
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()


compare_policies(task_name="left_turn", controller="pure_pursuit", delay_s=1.5, seed=0)

In [ ]:
# --- Export sweep results for later plotting/sharing ---

raise ValueError("stop")

if pd is not None:
    out_csv = REPO / "logs" / "toy_delayed_vlm_sweep.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print("wrote:", out_csv)